In [2]:
import pandas as pd
import numpy as np
import pickle
import os

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [4]:
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed, skipping that model. Run: pip install xgboost")
 
os.makedirs("model", exist_ok=True)

In [6]:


DATA_PATH = "loan_data.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(df.head())

Loaded 45000 rows, 14 columns
   person_age person_gender person_education  person_income  person_emp_exp  \
0        22.0        female           Master        71948.0               0   
1        21.0        female      High School        12282.0               0   
2        25.0        female      High School        12438.0               3   
3        23.0        female         Bachelor        79753.0               0   
4        24.0          male           Master        66135.0               1   

  person_home_ownership  loan_amnt loan_intent  loan_int_rate  \
0                  RENT    35000.0    PERSONAL          16.02   
1                   OWN     1000.0   EDUCATION          11.14   
2              MORTGAGE     5500.0     MEDICAL          12.87   
3                  RENT    35000.0     MEDICAL          15.23   
4                  RENT    35000.0     MEDICAL          14.27   

   loan_percent_income  cb_person_cred_hist_length  credit_score  \
0                 0.49              

In [7]:
# CLEAN DATA
df = df.drop_duplicates()
 
# Fill numeric missing values with median, categorical with mode
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
 
target_col = "loan_status"
if target_col in num_cols:
    num_cols.remove(target_col)
 
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
 
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
 
print("\nClass balance:")
print(df[target_col].value_counts(normalize=True))


Class balance:
loan_status
0    0.777778
1    0.222222
Name: proportion, dtype: float64


In [8]:
# ENCODE CATEGORICAL COLUMNS 

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

In [9]:
# 4-> SPLIT FEATURES / TARGET

X = df.drop(columns=[target_col])
y = df[target_col]
 
feature_names = X.columns.tolist()
print(f"\nFeatures used ({len(feature_names)}): {feature_names}")
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


Features used (13): ['person_age', 'person_gender', 'person_education', 'person_income', 'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'previous_loan_defaults_on_file']


In [10]:
# 5. SCALE NUMERIC FEATURES

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
# 6. TRAIN AND COMPARE MODELS

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
}
if HAS_XGB:
    models["XGBoost"] = XGBClassifier(
        n_estimators=200, eval_metric="logloss", random_state=42
    )
 
results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    proba = model.predict_proba(X_test_scaled)[:, 1]
 
    results[name] = {
        "model": model,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, proba),
    }
 
print("\n===== Model Comparison =====")
for name, r in results.items():
    print(f"{name:20s} | acc={r['accuracy']:.3f}  prec={r['precision']:.3f}  "
          f"rec={r['recall']:.3f}  f1={r['f1']:.3f}  roc_auc={r['roc_auc']:.3f}")


===== Model Comparison =====
LogisticRegression   | acc=0.897  prec=0.778  rec=0.750  f1=0.763  roc_auc=0.951
RandomForest         | acc=0.929  prec=0.890  rec=0.776  f1=0.830  roc_auc=0.974
XGBoost              | acc=0.937  prec=0.886  rec=0.821  f1=0.852  roc_auc=0.979


In [12]:
# 7. PICK THE BEST MODEL AND SAVE EVERYTHING

best_name = max(results, key=lambda n: results[n]["roc_auc"])
best_model = results[best_name]["model"]
print(f"\nBest model: {best_name} (roc_auc={results[best_name]['roc_auc']:.3f})")
 
with open("model/model.pkl", "wb") as f:
    pickle.dump(best_model, f)
 
with open("model/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
 
with open("model/encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)
 
with open("model/feature_names.pkl", "wb") as f:
    pickle.dump(feature_names, f)
 
print("\nSaved model.pkl, scaler.pkl, encoders.pkl, feature_names.pkl to model/")


Best model: XGBoost (roc_auc=0.979)

Saved model.pkl, scaler.pkl, encoders.pkl, feature_names.pkl to model/
